In [ ]:
# Install OS audio dependencies
!apt-get update && apt-get install -y libsndfile1 ffmpeg

# Install the Python ML and Audio libraries
!pip install funasr modelscope opensmile librosa openai torch torchaudio soundfile numpy

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
ffmpeg is already the newest ve

In [ ]:
import os

# Set OPENAI_API_KEY in your environment before running cells that call OpenAI.
# Example: export OPENAI_API_KEY="..."
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


OpenAI-backed analysis cells read `OPENAI_API_KEY` from the environment when available.


In [ ]:
import re
import numpy as np
import opensmile
import librosa
import torch
import json
import logging
import os
from openai import OpenAI
from collections import Counter
from typing import Dict, Any, Tuple
from funasr import AutoModel

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("AffectiveMemoryEngine")

class AffectiveMemoryExtractor:
    """
    Post-call background worker that extracts paralinguistic features and emotional
    state from raw audio using a dual-engine approach (SenseVoice + openSMILE),
    now enhanced with Lexical Sentiment Analysis for Contradiction Detection.
    """

    def __init__(self, device: str = "cuda:0"):
        self.device = device if torch.cuda.is_available() else "cpu"
        logger.info(f"Initializing Affective Memory Extractor on {self.device}...")

        # 1. Initialize SenseVoice (FunAudioLLM)
        try:
            self.sense_voice = AutoModel(
                model="iic/SenseVoiceSmall",
                trust_remote_code=True,
                vad_model="fsmn-vad",
                vad_kwargs={"max_single_segment_time": 30000},
                device=self.device
            )
            logger.info("SenseVoice engine loaded successfully.")
        except Exception as e:
            logger.error(f"Failed to load SenseVoice: {e}")
            raise

        # 2. Initialize openSMILE for deep biometrics
        self.smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.eGeMAPSv02,
            feature_level=opensmile.FeatureLevel.Functionals,
        )
        logger.info("openSMILE engine loaded successfully.")

        # 3. Initialize OpenAI Client for Deep Lexical/Contradiction Analysis
        # This replaces VADER to catch complex sarcasm, psychopathy, and masked grief.
        try:
            self.openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
            logger.info("OpenAI LLM engine loaded successfully.")
        except Exception as e:
            logger.warning(f"Failed to load OpenAI Client: {e}")
            self.openai_client = None

    def _parse_sensevoice_output(self, raw_text: str) -> Tuple[str, str, list]:
        """
        SenseVoice outputs rich transcription with embedded tags.
        Example: "<|zh|><|NEUTRAL|><|Speech|> Hello there <|Laughter|>"
        """
        emotions = re.findall(r'<\|(HAPPY|SAD|ANGRY|NEUTRAL)\|>', raw_text)
        if emotions:
            dominant_emotion = Counter([e.lower() for e in emotions]).most_common(1)[0][0]
        else:
            dominant_emotion = "neutral"

        events = re.findall(r'<\|(Laughter|Applause|Cough|Sneeze|Cry|Music)\|>', raw_text)
        events = [e.lower() for e in events]

        clean_text = re.sub(r'<\|.*?\|>', '', raw_text).strip()

        return clean_text, dominant_emotion, events

    def extract_deep_acoustics(self, audio_path: str) -> Dict[str, float]:
        """
        Uses openSMILE to extract low-level physiological speech parameters.
        """
        try:
            features_df = self.smile.process_file(audio_path)

            def safe_get(base_name: str) -> float:
                cols = [c for c in features_df.columns if base_name.lower() in c.lower()]
                return float(features_df[cols[0]].iloc[0]) if cols else 0.0

            acoustics = {
                "loudness_mean": safe_get('Loudness_sma3'),
                "pitch_mean": safe_get('F0semitoneFrom27.5Hz'),
                "jitter_local": safe_get('jitterLocal'),
                "shimmer_local": safe_get('shimmerLocal'),
                "speaking_rate": safe_get('equivalentSoundLevel')
            }
            return acoustics
        except Exception as e:
            logger.warning(f"openSMILE extraction failed: {e}")
            return {}

    def extract_temporal_and_pitch_dynamics(self, audio_path: str) -> Dict[str, Any]:
        """
        Uses librosa to extract temporal features (silence ratio, pauses)
        and pitch contour (trajectory) to detect hesitation, grief, or vocal fry.
        """
        try:
            # Load audio for temporal analysis
            y, sr = librosa.load(audio_path, sr=16000)

            # 1. Temporal Analysis (Silence & Pauses)
            # top_db=30: Anything 30dB below peak is considered "silence/dead air"
            non_silent_intervals = librosa.effects.split(y, top_db=30)
            total_duration = librosa.get_duration(y=y, sr=sr)

            non_silent_duration = sum([(end - start) / sr for start, end in non_silent_intervals])
            silence_ratio = 1.0 - (non_silent_duration / total_duration) if total_duration > 0 else 0.0
            pause_count = len(non_silent_intervals) - 1 if len(non_silent_intervals) > 1 else 0

            # 2. Pitch Trajectory (Falling/Rising/Flat)
            # Extract fundamental frequency (f0) using the YIN algorithm
            f0 = librosa.yin(y, fmin=50, fmax=500)
            f0_valid = f0[f0 > 0]

            pitch_trajectory = "flat/steady"
            if len(f0_valid) > 20:
                # Compare the first 30% vs the last 30% of the speech pitch
                start_segment = f0_valid[:int(len(f0_valid)*0.3)]
                end_segment = f0_valid[-int(len(f0_valid)*0.3):]
                pitch_drop = np.mean(start_segment) - np.mean(end_segment)

                if pitch_drop > 15:
                    pitch_trajectory = "falling (potential vocal fry, defeat, or fatigue)"
                elif pitch_drop < -15:
                    pitch_trajectory = "rising (seeking validation, questioning, or escalating panic)"

            return {
                "silence_ratio": round(silence_ratio, 3),
                "pause_count": pause_count,
                "pitch_trajectory": pitch_trajectory
            }
        except Exception as e:
            logger.warning(f"Librosa temporal extraction failed: {e}")
            return {"silence_ratio": 0.0, "pause_count": 0, "pitch_trajectory": "unknown"}

    def _evaluate_contradiction_llm(self, transcript: str, acoustic_emotion: str, acoustics: Dict[str, Any]) -> Dict[str, Any]:
        """
        Uses an LLM to deeply analyze the transcript against the acoustic findings.
        Returns a JSON-structured evaluation of the true affective state.
        """
        if not self.openai_client or not transcript.strip():
            return {"lexical_sentiment": "neutral", "final_affective_state": acoustic_emotion, "reasoning": "LLM offline."}

        system_prompt = (
            "You are an expert psychological profiler and paralinguistics engine. "
            "You will evaluate a transcript alongside deep acoustic biometrics.\n"
            "- Jitter > 0.05 indicates vocal cord tension (stress/suppression).\n"
            "- High Silence Ratio (> 0.20) & frequent pauses indicate grief, heavy cognitive load, or hesitation.\n"
            "- Pitch Trajectory (Falling) indicates defeat, giving up, or apathy (vocal fry).\n"
            "Detect contradictions between the text and how they sound, and output the TRUE underlying affective state.\n\n"
            "Output strictly as JSON:\n"
            "{\n"
            '  "lexical_sentiment": "positive|negative|neutral",\n'
            '  "final_affective_state": "e.g., psychopathic_threat, cynical_or_masking_grief, tense_suppressed, genuine_joy",\n'
            '  "reasoning": "1 sentence explaining why"\n'
            "}"
        )

        user_prompt = (
             f"Transcript: '{transcript}'\n"
             f"Acoustic Emotion: {acoustic_emotion}\n"
             f"Jitter (Tension): {acoustics.get('jitter_local', 0)}\n"
             f"Silence Ratio: {acoustics.get('silence_ratio', 0)}\n"
             f"Pause Count: {acoustics.get('pause_count', 0)}\n"
             f"Pitch Trajectory: {acoustics.get('pitch_trajectory', 'unknown')}"
        )

        try:
            response = self.openai_client.chat.completions.create(
                model="gpt-4o-mini",
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            logger.error(f"LLM Contradiction Evaluation failed: {e}")
            return {"lexical_sentiment": "neutral", "final_affective_state": acoustic_emotion, "reasoning": "Error"}

    def process_call_audio(self, audio_path: str, user_id: str, call_id: str) -> str:
        """
        Main pipeline function running dual acoustic/lexical contradiction detection.
        """
        logger.info(f"Processing audio for Call: {call_id} | User: {user_id}")

        # 1. Acoustic Extraction (SenseVoice)
        sv_results = self.sense_voice.generate(
            input=audio_path,
            language="auto",
            use_itn=True,
            batch_size=64
        )
        full_raw_text = " ".join([res['text'] for res in sv_results])
        transcript, acoustic_emotion, audio_events = self._parse_sensevoice_output(full_raw_text)

        # 2. Deep Biometric Extraction (openSMILE)
        deep_acoustics = self.extract_deep_acoustics(audio_path)

        # 3. Temporal & Contour Dynamics (Librosa)
        temporal_dynamics = self.extract_temporal_and_pitch_dynamics(audio_path)

        # Merge all acoustic and temporal data into a single comprehensive profile
        all_acoustics = {**deep_acoustics, **temporal_dynamics}

        # 4. LLM Lexical Sentiment & Contradiction Engine
        # The LLM now has access to Jitter, Silence Ratios, and Pitch trajectories!
        llm_evaluation = self._evaluate_contradiction_llm(transcript, acoustic_emotion, all_acoustics)

        # 5. Construct Memory Payload
        memory_payload = {
            "metadata": {
                "user_id": user_id,
                "call_id": call_id,
            },
            "paralinguistics": {
                "base_acoustic_emotion": acoustic_emotion,
                "lexical_sentiment": llm_evaluation.get("lexical_sentiment", "neutral"),
                "llm_reasoning": llm_evaluation.get("reasoning", ""),
                "final_affective_state": llm_evaluation.get("final_affective_state", acoustic_emotion),
                "detected_events": list(set(audio_events)),
                "acoustic_biometrics": all_acoustics
            },
            "transcript": transcript
        }

        logger.info(f"Successfully generated Paralinguistic Memory Profile for {call_id}")
        return json.dumps(memory_payload, indent=2)

# ==============================================================================
# Example Usage:
# ==============================================================================
if __name__ == "__main__":
    extractor = AffectiveMemoryExtractor()
    sample_audio_path = "./post_call_records/test3.wav"

    import os
    if os.path.exists(sample_audio_path):
        result_json = extractor.process_call_audio(
            audio_path=sample_audio_path,
            user_id="usr_778",
            call_id="call_991"
        )
        print("Final Affective Payload for DB:\n", result_json)
    else:
        print(f"Waiting for audio files at {sample_audio_path}...")

funasr version: 1.3.14.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
You are using the latest version of funasr-1.3.14


2026-07-11 05:55:50,000 | INFO    | modelscope_hub.download | Downloading 20 files from iic/SenseVoiceSmall@master


Downloading:   0%|          | 0/20 [00:00<?, ?file/s]

Loading remote code failed: model, No module named 'model'


2026-07-11 05:56:06,424 | INFO    | modelscope_hub.download | Downloading 8 files from iic/speech_fsmn_vad_zh-cn-16k-common-pytorch@master


...

struct.png:   0%|          | 0.00/27.9k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

vad_example.wav:   0%|          | 0.00/2.26M [00:00<?, ?B/s]

100%|██████████| 1/1 [00:00<00:00,  2.01it/s]
{'load_data': '0.000', 'extract_feat': '0.008', 'forward': '0.498', 'batch_size': '1', 'rtf': '0.553'}, : 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]
rtf_avg: 0.553: 100%|██████████| 1/1 [00:00<00:00,  1.98it/s]

100%|██████████| 1/1 [00:00<00:00,  2.01it/s]
{'load_data': '0.000', 'extract_feat': '0.007', 'forward': '0.497', 'batch_size': '1', 'rtf': '0.517'}, : 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]
rtf_avg: 0.517: 100%|██████████| 1/1 [00:00<00:00,  1.97it/s]

100%|██████████| 1/1 [00:00<00:00,  2.08it/s]
{'load_data': '0.000', 'extract_feat': '0.008', 'forward': '0.481', 'batch_size': '1', 'rtf': '0.422'}, : 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]
rtf_avg: 0.422: 100%|██████████| 1/1 [00:00<00:00,  2.04it/s]

100%|██████████| 1/1 [00:00<00:00,  1.85it/s]
{'load_data': '0.000', 'extract_feat': '0.011', 'forward': '0.541', 'batch_size': '1', 'rtf': '0.392'}, : 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]
rtf_avg: 0.392: 100

Final Affective Payload for DB:
 {
  "metadata": {
    "user_id": "usr_778",
    "call_id": "call_991"
  },
  "paralinguistics": {
    "base_acoustic_emotion": "happy",
    "lexical_sentiment": "negative",
    "llm_reasoning": "The speaker expresses deep grief and cynicism about their relationship with their mother, despite the acoustic emotion suggesting happiness, indicating a disconnect between their verbal expression and true emotional state.",
    "final_affective_state": "cynical_or_masking_grief",
    "detected_events": [],
    "acoustic_biometrics": {
      "loudness_mean": 0.6302699446678162,
      "pitch_mean": 25.227800369262695,
      "jitter_local": 0.04426059499382973,
      "shimmer_local": 1.4594980478286743,
      "speaking_rate": -29.60452651977539,
      "silence_ratio": 0.221,
      "pause_count": 46,
      "pitch_trajectory": "rising (seeking validation, questioning, or escalating panic)"
    }
  },
  "transcript": "My mom died and all I got was this free chro. You

In [ ]:
import re
import numpy as np
import opensmile
import librosa
import torch
import json
import logging
import os
from openai import OpenAI
from collections import Counter
from typing import Dict, Any, Tuple
from funasr import AutoModel

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("AffectiveMemoryEngine")

class AffectiveMemoryExtractor:
    """
    Post-call background worker that extracts paralinguistic features and emotional
    state from raw audio using a dual-engine approach (SenseVoice + openSMILE),
    now enhanced with Lexical Sentiment Analysis for Contradiction Detection.
    """

    def __init__(self, device: str = "cuda:0"):
        self.device = device if torch.cuda.is_available() else "cpu"
        logger.info(f"Initializing Affective Memory Extractor on {self.device}...")

        # 1. Initialize SenseVoice (FunAudioLLM)
        try:
            self.sense_voice = AutoModel(
                model="iic/SenseVoiceSmall",
                trust_remote_code=True,
                vad_model="fsmn-vad",
                vad_kwargs={"max_single_segment_time": 30000},
                device=self.device
            )
            logger.info("SenseVoice engine loaded successfully.")
        except Exception as e:
            logger.error(f"Failed to load SenseVoice: {e}")
            raise

        # 2. Initialize openSMILE for deep biometrics
        self.smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.eGeMAPSv02,
            feature_level=opensmile.FeatureLevel.Functionals,
        )
        logger.info("openSMILE engine loaded successfully.")

        # 3. Initialize OpenAI Client for Deep Lexical/Contradiction Analysis
        # This replaces VADER to catch complex sarcasm, psychopathy, and masked grief.
        try:
            self.openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
            logger.info("OpenAI LLM engine loaded successfully.")
        except Exception as e:
            logger.warning(f"Failed to load OpenAI Client: {e}")
            self.openai_client = None

    def _parse_sensevoice_output(self, raw_text: str) -> Tuple[str, str, list]:
        """
        SenseVoice outputs rich transcription with embedded tags.
        Example: "<|zh|><|NEUTRAL|><|Speech|> Hello there <|Laughter|>"
        """
        emotions = re.findall(r'<\|(HAPPY|SAD|ANGRY|NEUTRAL)\|>', raw_text)
        if emotions:
            dominant_emotion = Counter([e.lower() for e in emotions]).most_common(1)[0][0]
        else:
            dominant_emotion = "neutral"

        events = re.findall(r'<\|(Laughter|Applause|Cough|Sneeze|Cry|Music)\|>', raw_text)
        events = [e.lower() for e in events]

        clean_text = re.sub(r'<\|.*?\|>', '', raw_text).strip()

        return clean_text, dominant_emotion, events

    def extract_deep_acoustics(self, audio_path: str) -> Dict[str, float]:
        """
        Uses openSMILE to extract low-level physiological speech parameters.
        """
        try:
            features_df = self.smile.process_file(audio_path)

            def safe_get(base_name: str) -> float:
                cols = [c for c in features_df.columns if base_name.lower() in c.lower()]
                return float(features_df[cols[0]].iloc[0]) if cols else 0.0

            acoustics = {
                "loudness_mean": safe_get('Loudness_sma3'),
                "pitch_mean": safe_get('F0semitoneFrom27.5Hz'),
                "jitter_local": safe_get('jitterLocal'),
                "shimmer_local": safe_get('shimmerLocal'),
                "speaking_rate": safe_get('equivalentSoundLevel')
            }
            return acoustics
        except Exception as e:
            logger.warning(f"openSMILE extraction failed: {e}")
            return {}

    def extract_temporal_and_pitch_dynamics(self, audio_path: str) -> Dict[str, Any]:
        """
        Uses librosa to extract temporal features (silence ratio, pauses)
        and pitch contour (trajectory) to detect hesitation, grief, or vocal fry.
        """
        try:
            # Load audio for temporal analysis
            y, sr = librosa.load(audio_path, sr=16000)

            # 1. Temporal Analysis (Silence & Pauses)
            # top_db=30: Anything 30dB below peak is considered "silence/dead air"
            non_silent_intervals = librosa.effects.split(y, top_db=30)
            total_duration = librosa.get_duration(y=y, sr=sr)

            non_silent_duration = sum([(end - start) / sr for start, end in non_silent_intervals])
            silence_ratio = 1.0 - (non_silent_duration / total_duration) if total_duration > 0 else 0.0
            pause_count = len(non_silent_intervals) - 1 if len(non_silent_intervals) > 1 else 0

            # 2. Pitch Trajectory (Falling/Rising/Flat)
            # Extract fundamental frequency (f0) using the YIN algorithm
            f0 = librosa.yin(y, fmin=50, fmax=500)
            f0_valid = f0[f0 > 0]

            pitch_trajectory = "flat/steady"
            if len(f0_valid) > 20:
                # Compare the first 30% vs the last 30% of the speech pitch
                start_segment = f0_valid[:int(len(f0_valid)*0.3)]
                end_segment = f0_valid[-int(len(f0_valid)*0.3):]
                pitch_drop = np.mean(start_segment) - np.mean(end_segment)

                if pitch_drop > 15:
                    pitch_trajectory = "falling (potential vocal fry, defeat, or fatigue)"
                elif pitch_drop < -15:
                    pitch_trajectory = "rising (seeking validation, questioning, or escalating panic)"

            return {
                "silence_ratio": round(silence_ratio, 3),
                "pause_count": pause_count,
                "pitch_trajectory": pitch_trajectory
            }
        except Exception as e:
            logger.warning(f"Librosa temporal extraction failed: {e}")
            return {"silence_ratio": 0.0, "pause_count": 0, "pitch_trajectory": "unknown"}

    def _evaluate_contradiction_llm(self, transcript: str, acoustic_emotion: str, acoustics: Dict[str, Any]) -> Dict[str, Any]:
        """
        Uses an LLM to deeply analyze the transcript against the acoustic findings.
        Returns a JSON-structured evaluation of the true affective state.
        """
        if not self.openai_client or not transcript.strip():
            return {"lexical_sentiment": "neutral", "final_affective_state": acoustic_emotion, "reasoning": "LLM offline."}

        system_prompt = (
            "You are an expert psychological profiler and paralinguistics engine. "
            "You will evaluate a transcript alongside deep acoustic biometrics.\n"
            "- Jitter > 0.05 indicates vocal cord tension (stress/suppression).\n"
            "- High Silence Ratio (> 0.20) & frequent pauses indicate grief, heavy cognitive load, or hesitation.\n"
            "- Pitch Trajectory (Falling) indicates defeat, giving up, or apathy (vocal fry).\n"
            "Detect contradictions between the text and how they sound, and output the TRUE underlying affective state.\n\n"
            "Output strictly as JSON:\n"
            "{\n"
            '  "lexical_sentiment": "positive|negative|neutral",\n'
            '  "final_affective_state": "e.g., psychopathic_threat, cynical_or_masking_grief, tense_suppressed, genuine_joy",\n'
            '  "reasoning": "1 sentence explaining why"\n'
            "}"
        )

        user_prompt = (
             f"Transcript: '{transcript}'\n"
             f"Acoustic Emotion: {acoustic_emotion}\n"
             f"Jitter (Tension): {acoustics.get('jitter_local', 0)}\n"
             f"Silence Ratio: {acoustics.get('silence_ratio', 0)}\n"
             f"Pause Count: {acoustics.get('pause_count', 0)}\n"
             f"Pitch Trajectory: {acoustics.get('pitch_trajectory', 'unknown')}"
        )

        try:
            response = self.openai_client.chat.completions.create(
                model="gpt-4o-mini",
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            logger.error(f"LLM Contradiction Evaluation failed: {e}")
            return {"lexical_sentiment": "neutral", "final_affective_state": acoustic_emotion, "reasoning": "Error"}

    def process_call_audio(self, audio_path: str, user_id: str, call_id: str) -> str:
        """
        Main pipeline function running dual acoustic/lexical contradiction detection.
        """
        logger.info(f"Processing audio for Call: {call_id} | User: {user_id}")

        # 1. Acoustic Extraction (SenseVoice)
        sv_results = self.sense_voice.generate(
            input=audio_path,
            language="auto",
            use_itn=True,
            batch_size=64
        )
        full_raw_text = " ".join([res['text'] for res in sv_results])
        transcript, acoustic_emotion, audio_events = self._parse_sensevoice_output(full_raw_text)

        # 2. Deep Biometric Extraction (openSMILE)
        deep_acoustics = self.extract_deep_acoustics(audio_path)

        # 3. Temporal & Contour Dynamics (Librosa)
        temporal_dynamics = self.extract_temporal_and_pitch_dynamics(audio_path)

        # Merge all acoustic and temporal data into a single comprehensive profile
        all_acoustics = {**deep_acoustics, **temporal_dynamics}

        # 4. LLM Lexical Sentiment & Contradiction Engine
        # The LLM now has access to Jitter, Silence Ratios, and Pitch trajectories!
        llm_evaluation = self._evaluate_contradiction_llm(transcript, acoustic_emotion, all_acoustics)

        # 5. Construct Memory Payload
        memory_payload = {
            "metadata": {
                "user_id": user_id,
                "call_id": call_id,
            },
            "paralinguistics": {
                "base_acoustic_emotion": acoustic_emotion,
                "lexical_sentiment": llm_evaluation.get("lexical_sentiment", "neutral"),
                "llm_reasoning": llm_evaluation.get("reasoning", ""),
                "final_affective_state": llm_evaluation.get("final_affective_state", acoustic_emotion),
                "detected_events": list(set(audio_events)),
                "acoustic_biometrics": all_acoustics
            },
            "transcript": transcript
        }

        logger.info(f"Successfully generated Paralinguistic Memory Profile for {call_id}")
        return json.dumps(memory_payload, indent=2)

# ==============================================================================
# Example Usage:
# ==============================================================================
if __name__ == "__main__":
    extractor = AffectiveMemoryExtractor()
    sample_audio_path = "./post_call_records/test2.wav"

    import os
    if os.path.exists(sample_audio_path):
        result_json = extractor.process_call_audio(
            audio_path=sample_audio_path,
            user_id="usr_778",
            call_id="call_991"
        )
        print("Final Affective Payload for DB:\n", result_json)
    else:
        print(f"Waiting for audio files at {sample_audio_path}...")

funasr version: 1.3.14.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
You are using the latest version of funasr-1.3.14


2026-07-11 05:57:38,221 | INFO    | modelscope_hub.download | Downloading 20 files from iic/SenseVoiceSmall@master


Downloading:   0%|          | 0/20 [00:00<?, ?file/s]

Loading remote code failed: model, No module named 'model'


2026-07-11 05:57:48,162 | INFO    | modelscope_hub.download | Downloading 8 files from iic/speech_fsmn_vad_zh-cn-16k-common-pytorch@master


Downloading:   0%|          | 0/8 [00:00<?, ?file/s]

100%|██████████| 1/1 [00:04<00:00,  4.92s/it]
{'load_data': '0.000', 'extract_feat': '0.034', 'forward': '4.922', 'batch_size': '1', 'rtf': '0.193'}, : 100%|██████████| 1/1 [00:04<00:00,  4.92s/it]
rtf_avg: 0.193: 100%|██████████| 1/1 [00:04<00:00,  4.94s/it]

100%|██████████| 1/1 [00:07<00:00,  7.16s/it]
{'load_data': '0.000', 'extract_feat': '0.036', 'forward': '7.161', 'batch_size': '1', 'rtf': '0.239'}, : 100%|██████████| 1/1 [00:07<00:00,  7.16s/it]
rtf_avg: 0.239: 100%|██████████| 1/1 [00:07<00:00,  7.18s/it]
rtf_avg: 0.219, time_speech:  55.496, time_escape: 12.129: 100%|██████████| 1/1 [00:12<00:00, 12.27s/it]


Final Affective Payload for DB:
 {
  "metadata": {
    "user_id": "usr_778",
    "call_id": "call_991"
  },
  "paralinguistics": {
    "base_acoustic_emotion": "neutral",
    "lexical_sentiment": "negative",
    "llm_reasoning": "The negative content of the transcript, combined with the elevated jitter indicating vocal tension and a flat pitch trajectory, suggests a suppressed emotional state despite the neutral acoustic emotion.",
    "final_affective_state": "tense_suppressed",
    "detected_events": [],
    "acoustic_biometrics": {
      "loudness_mean": 1.015892744064331,
      "pitch_mean": 18.183666229248047,
      "jitter_local": 0.0678616613149643,
      "shimmer_local": 1.466506838798523,
      "speaking_rate": -24.11100959777832,
      "silence_ratio": 0.098,
      "pause_count": 9,
      "pitch_trajectory": "flat/steady"
    }
  },
  "transcript": "Give me hes joking want to know how I got these scars My father was a drinker in a fiend and one night he goes off crazier than 